# Typed messages in a standalone runtime: practice

Define Message → define RoutedAgent handler → register type → start runtime → send to AgentId → stop/close → compose multiple message handlers.

Use this with [the walkthrough](walkthrough.md).

**Source:**

- [5_autogen/3_lab3_autogen_core.ipynb](../../../../../5_autogen/3_lab3_autogen_core.ipynb)

**Run one cell at a time:** predict → run with Shift+Enter → explain → change one thing. Select the root project .venv kernel.

**Offline adaptation:** code below uses fictional data or local libraries. It never executes the linked source, reads a credential file, sends messages, starts a UI/server, or changes the course databases. Simulations are labeled; they do not verify external integrations.

## 1. Typed messages and agent identities

A runtime delivers a message to an identified handler. The handler owns its work. This dictionary-based simulation exposes type/key routing without starting AutoGen or network listeners.

**Predict:** Will an unregistered type resolve?

In [ ]:
from dataclasses import dataclass
@dataclass
class Message:
    content: str
registry = {("simple_agent", "default"): lambda message: Message("Received: " + message.content)}
recipient = ("simple_agent", "default")  # Agent type and instance key.
response = registry[recipient](Message("Hello"))
print(response.content)
assert response.content == "Received: Hello"

**Check:** No. Registration must precede delivery. The simulation does not verify an AutoGen runtime.

**Change one thing:** Register a second handler and send the response to it. Explain which code routes and which code does the work.

## 2. Run a real standalone AutoGen handler without a model

A local runtime can route ordinary Python messages without AgentChat or an LLM. Register the type, start the runtime, send to its type/key identity, then stop and close.

**Predict:** Which handler receives the message sent to simple_agent/default?

In [ ]:
from dataclasses import dataclass
from autogen_core import AgentId, MessageContext, RoutedAgent, message_handler, SingleThreadedAgentRuntime
@dataclass
class PracticeMessage:
    content: str
class SimplePracticeAgent(RoutedAgent):
    def __init__(self):
        super().__init__("Offline handler")
    @message_handler
    async def on_message(self, message: PracticeMessage, ctx: MessageContext) -> PracticeMessage:
        return PracticeMessage("Received: " + message.content)
runtime = SingleThreadedAgentRuntime()
await SimplePracticeAgent.register(runtime, "simple_agent", lambda: SimplePracticeAgent())
runtime.start()
try:
    response = await runtime.send_message(PracticeMessage("Hello"), AgentId("simple_agent", "default"))
    print(response.content)
    assert response.content == "Received: Hello"
finally:
    await runtime.stop()
    await runtime.close()

**Check:** SimplePracticeAgent. This tests local routing, not gRPC distribution or hosted model behavior.

**Change one thing:** Change the reply text and send another message. Explain why registration must precede delivery and why close is still necessary.

## 3. Messages: instructions, earlier turns, and the new question

The model request is a list of messages. A role says who is speaking; content holds the text. The system instruction and prior conversation serve different purposes.

**Predict:** How many messages are sent on the follow-up turn?

In [ ]:
instructions = "Explain Python with small examples."
history = [
    {"role": "user", "content": "What is a list?"},
    {"role": "assistant", "content": "An ordered collection of values."},
]
# List addition preserves the order: instruction, old turns, new question.
messages = [{"role": "system", "content": instructions}] + history + [
    {"role": "user", "content": "How is a dictionary different?"}
]
print([item["role"] for item in messages])
assert len(messages) == 4

**Check:** Four: system, user, assistant, user. With empty history, only two remain.

**Change one thing:** Set history to an empty list. Explain what is lost while the system instruction remains.

## 4. Team turns need a termination condition

A round-robin team alternates speakers. A marker can end the conversation; a maximum turn count stops it even if the marker never appears.

**Predict:** At what turn does this conversation stop?

In [ ]:
scripted = [("writer", "Initial draft"), ("evaluator", "Add an example"),
            ("writer", "Revised draft with an example"), ("evaluator", "APPROVE")]
seen = []
for speaker, content in scripted[:6]:
    seen.append((speaker, content))
    if "APPROVE" in content:
        break
print(seen)
assert len(seen) == 4

**Check:** Turn four. Text-based termination is a rule, not proof that the work is complete.

**Change one thing:** Remove APPROVE and add a finite fallback when no approval occurs. Discuss accidental marker mentions.

## 5. Await vs. concurrent work

Calling an async function creates a coroutine. Await executes/waits for it. gather schedules independent coroutines and returns results in input order. Notebook cells support top-level await.

**Predict:** Will results follow completion order or input order?

In [ ]:
import asyncio
async def work(label, delay):
    await asyncio.sleep(delay)  # Yield control while waiting.
    return label
results = await asyncio.gather(work("A", 0.002), work("B", 0))
print(results)
assert results == ["A", "B"]

**Check:** Input order. In a standalone script, wrap work in async def main and call asyncio.run(main()); do not do that inside a running notebook loop.

**Change one thing:** Run the two functions with separate awaits. Explain why a dependent stage cannot start until it has its input.

## Rebuild from memory

1. Recreate the main data-flow step in a new scratch cell.
2. Add one case that should fail and identify its exception/stop condition.
3. Compare your version with the linked source cells or functions.
4. Explain where a real model or service would enter, and what it could change.

For deliberate original-source execution, see the shared [configuration guide](../../../configuration.md). All provider/transport integrations remain UNVERIFIED at runtime.